# NeuroRAG Ablation Study on MMLU

This notebook allows you to disable (ablate) almost any component of the NeuroRAG pipeline and evaluate the effect on MMLU accuracy.

In [48]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import re
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass
from pydantic import BaseModel, Field

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from neurorag.neurorag import NeuroRAG


## Disable warnings

In [49]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

In [50]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Ablation Config
Set any component to False to disable it in the pipeline.

In [51]:
ablation_config: dict[str, bool] = {
    'step_back': True,
    'query_rewriting': True,
    'decomposition': True,
    'hyde': True,
    'vector_store': True,
    'pubmed': True,
    'arxiv': True,
    'ncbi_protein': True,
    'ncbi_gene': True,
    'biorxiv': True,
    'medrxiv': True,
    'document_grading': True,
    'hallucination_grading': True,
    'answer_grading': True,
    'web_search': True,
}


## NeuroRAG Wrapper for Ablation
This class disables components according to the config above.

In [52]:
class NeuroRAGAblation(NeuroRAG):
    def __init__(self, ablation_config, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ablation_config = ablation_config

    def generate_step_back_query_node(self, state):
        if not self.ablation_config.get('step_back', True):
            return {'step_back_query': state['query']}
        return super().generate_step_back_query_node(state)

    def generate_rewritten_query_node(self, state):
        if not self.ablation_config.get('query_rewriting', True):
            return {'rewritten_query': state['query']}
        return super().generate_rewritten_query_node(state)

    def generate_subqueries_node(self, state):
        if not self.ablation_config.get('decomposition', True):
            return {'subqueries': []}
        return super().generate_subqueries_node(state)

    def generate_hyde_documents_node(self, state):
        if not self.ablation_config.get('hyde', True):
            return {'generated_documents': [state['query']]}
        return super().generate_hyde_documents_node(state)

    def vector_store_retriever_node(self, state):
        if not self.ablation_config.get('vector_store', True):
            return {'documents': []}
        return super().vector_store_retriever_node(state)

    def pub_med_retriever_node(self, state):
        if not self.ablation_config.get('pubmed', True):
            return {'documents': []}
        return super().pub_med_retriever_node(state)

    def arxiv_retriever_node(self, state):
        if not self.ablation_config.get('arxiv', True):
            return {'documents': []}
        return super().arxiv_retriever_node(state)

    def ncbi_protein_db_retriever_node(self, state):
        if not self.ablation_config.get('ncbi_protein', True):
            return {'documents': []}
        return super().ncbi_protein_db_retriever_node(state)

    def ncbi_gene_db_retriever_node(self, state):
        if not self.ablation_config.get('ncbi_gene', True):
            return {'documents': []}
        return super().ncbi_gene_db_retriever_node(state)

    def biorxiv_retriever_node(self, state):
        if not self.ablation_config.get('biorxiv', True):
            return {'documents': []}
        return super().biorxiv_retriever_node(state)

    def medrxiv_retriever_node(self, state):
        if not self.ablation_config.get('medrxiv', True):
            return {'documents': []}
        return super().medrxiv_retriever_node(state)

    def grade_documents_node(self, state):
        if not self.ablation_config.get('document_grading', True):
            # Bypass grading, just pass all documents through
            return {'documents': state['documents'], 'web_search': False}
        return super().grade_documents_node(state)

    def grade_generation_node(self, state):
        if not self.ablation_config.get('hallucination_grading', True) and not self.ablation_config.get('answer_grading', True):
            return 'useful'
        if not self.ablation_config.get('hallucination_grading', True):
            # Only answer grading
            query = state['query']
            generation = state['generation']
            try:
                grade = self.answer_grade_chain.invoke(query, generation)
            except Exception:
                grade = 'no'
            return 'useful' if grade == 'yes' else 'not useful'
        if not self.ablation_config.get('answer_grading', True):
            # Only hallucination grading
            documents = state['documents']
            generation = state['generation']
            try:
                context = (
                    '\n\n' + '\n\n'.join(map(lambda doc: doc.page_content, documents)) + '\n\n'
                )
                grade = self.hallucinations_chain.invoke(generation, context)
            except Exception:
                grade = 'no'
            return 'useful' if grade == 'yes' else 'not useful'
        return super().grade_generation_node(state)

    def web_search_node(self, state):
        if not self.ablation_config.get('web_search', True):
            return {'documents': [], 'web_results': []}
        return super().web_search_node(state)


## Setup MMLU tests

In [53]:
def extract_json(response):
  json_pattern = r'\{.*?\}'
  match = re.search(json_pattern, response, re.DOTALL)

  if match:
    return match.group().strip().replace('\\\\', '\\')

  return response

In [54]:
class RAGSchema(BaseModel):
  correct_answer: str = Field(description='Based on the question and the provided context, choose the most accurate letter among [A, B, C, D].')

rag_parser = PydanticOutputParser(pydantic_object=RAGSchema)

rag_template = """
You are a knowledgeable assistant with expertise in multiple domains.
Read the following question and context carefully.

1. Use the context and your domain knowledge to determine the correct answer.
2. Do any necessary reasoning internally—do not include your chain of thought in the output.
3. Provide only the one-letter answer (from [A, B, C, D]) in valid JSON format.

{format_instructions}

Question:
{query}

Context (review carefully):
{context}

Possible answers:
A. {a}
B. {b}
C. {c}
D. {d}
"""
prompt = PromptTemplate(
  template=rag_template,
  input_variables=['query', 'a', 'b', 'c', 'd', 'context'],
  partial_variables={'format_instructions': rag_parser.get_format_instructions()},
)

## MMLU Evaluation Function
(Copied from mmlu-evaluation.ipynb)

In [55]:
from datasets import load_dataset
letter_to_number = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
def eval_rag(app, mmlu_subset: str) -> float:
    dataset = load_dataset('cais/mmlu', mmlu_subset)
    test_df = dataset['test'].to_pandas()

    correct_answers_count = 0

    for index, row in tqdm(list(test_df.iterrows()), desc='Questions'):
        question = row['question']
        choices = row['choices']
        correct_answer = row['answer']

        prompt_with_choices = prompt.partial(
          a=choices[0],
          b=choices[1],
          c=choices[2],
          d=choices[3],
        )
        app.generation_prompt = prompt_with_choices
        llm_answer = app.invoke(question)

        llm_answer_letter = llm_answer['generation'].strip().lower()[0]

        if llm_answer_letter not in letter_to_number:
            continue

        llm_answer_num = letter_to_number[llm_answer_letter]
        if llm_answer_num == correct_answer:
            correct_answers_count += 1

    return correct_answers_count / len(test_df)


## Run Ablation Study
Try a few ablation settings and compare results.

In [56]:
ablation_settings = [
    ('All enabled', ablation_config.copy()),
    ('No step_back', {**ablation_config, 'step_back': False}),
    ('No query_rewriting', {**ablation_config, 'query_rewriting': False}),
    ('No decomposition', {**ablation_config, 'decomposition': False}),
    ('No hyde', {**ablation_config, 'hyde': False}),
    ('No document_grading', {**ablation_config, 'document_grading': False}),
    ('No hallucination_grading', {**ablation_config, 'hallucination_grading': False}),
    ('No answer_grading', {**ablation_config, 'answer_grading': False}),
    ('No web_search', {**ablation_config, 'web_search': False}),
]

results = []
for name, config in ablation_settings:
    print(f'Running: {name}')
    app = NeuroRAGAblation(config, model='llama3.1', debug=False)
    app.compile()
    acc = eval_rag(app, 'medical_genetics')
    results.append({'setting': name, 'accuracy': acc})
    print(f'{name}: {acc}')

pd.DataFrame(results)

Running: All enabled


Questions:   0%|          | 0/135 [01:26<?, ?it/s]


APIConnectionError: Connection error.